# Completing Random Forest Analysis
Assumption that appropriate separation and PCA has been completed.

## SETUP

In [86]:
import subprocess, sys, os, importlib, re
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import rf_functions 
importlib.reload(rf_functions)

parent_dir=os.path.dirname(os.getcwd())

doscar_files_dir=os.path.join(parent_dir, 'DOSCAR_files')

print(f"The base directory containing all the DOSCAR files is {doscar_files_dir}")

The base directory containing all the DOSCAR files is c:\Users\jespe\Desktop\[THESIS] Code\-ESPEJO-THESIS-DOS-ML-Model-\DOSCAR_files


### STEP 0 : LOAD THE ADSORPTION ENERGY DATA

In [87]:
adsorption_energy_df = pd.read_excel('Adsorption_energies_additives.xlsx')

adsorption_energy_df

,System,Ads.
0,"1,1,1,2-Tetrachloroethane",-0.40
1,"1,1,1,3,3,3-Hexafluoro-2-propanol",-0.52
2,"1,1,1-Trichloroethane",-0.49
3,"1,1,1-Trifluoro-2-propanol",-0.67
4,"1,1,2-trichloro-1,2,2-trifluoroethane",-0.45
...,...,...
285,Tetramethylurea,-1.01
286,Threonine,-0.69
287,Triethanolamine,-0.60
288,Vanillin,-0.53


In [88]:
# Initialize an empty list to store the results
transformed_data_dir = os.path.join(os.getcwd(),'first_data_set')

results = []

for filename in os.listdir(transformed_data_dir):
    filepath = os.path.join(transformed_data_dir, filename)

    # Load and process the data
    pca_data = rf_functions.load_pca_transformed_data(filepath)
    merged_data = rf_functions.adsorption_energy_for_dataset(pca_data, adsorption_energy_df)

    # Perform Random Forest and get mse and r2
    mse, r2 = rf_functions.perform_random_forest(merged_data)

    # Extract the energy range and cumulative variance ratio from the filename
    energy_range_match = re.search(r'\[([-+]?\d*\.\d+|\d+),([-+]?\d*\.\d+|\d+)\]', filename)
    cum_variance_match = re.search(r'var([\d\.]+)', filename)

    if energy_range_match and cum_variance_match:
        energy_range = f"{energy_range_match.group(1)},{energy_range_match.group(2)}"
        cum_variance = cum_variance_match.group(1)

        # Append the results to the list as a dictionary
        results.append({
            'energy_range': energy_range,
            'cumulative_variance_ratio': cum_variance,
            'num_principal_components': pca_data.shape[1] - 1,
            'mse': mse,
            'r2': r2
        })

# Convert the results list into a DataFrame
results_df = pd.DataFrame(results)

results_df

,energy_range,cumulative_variance_ratio,num_principal_components,mse,r2
0,"-10.00,10.00",0.80.,11,0.045928,-0.217497
1,"-10.00,10.00",0.90.,22,0.046700,-0.237974
2,"-11.44,4.12",0.80.,13,0.047856,-0.268623
3,"-11.44,4.12",0.90.,24,0.049917,-0.323242
4,"-5.00,5.00",0.80.,8,0.066527,-0.763556
5,"-5.00,5.00",0.90.,15,0.073673,-0.952980
